In [1]:
# !pip install python-dotenv
import os
from dotenv import load_dotenv

load_dotenv()  # loads variables from .env

print(os.getcwd())

C:\Users\sunit\Desktop\data_analysis\warehouse_bi_analysis


In [2]:
import pandas as pd
import numpy as np

In [3]:
# read csv
logistics_data = pd.read_csv("data/anonymized_logistics_picking_data.csv", sep=";")

# 1. Data Cleaning
## 1.1 Check structure abd datatypes

In [4]:
logistics_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 217963 entries, 0 to 217962
Data columns (total 18 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   location     217963 non-null  object 
 1   volume       217963 non-null  object 
 2   picker       217963 non-null  object 
 3   wm_vsola     217963 non-null  object 
 4   area         217963 non-null  object 
 5   warehouse    217963 non-null  object 
 6   calday       217963 non-null  object 
 7   calweek      217963 non-null  int64  
 8   calmonth     217963 non-null  object 
 9   calyear      217963 non-null  object 
 10  collli_id    217963 non-null  object 
 11  start_time   217869 non-null  object 
 12  finish_time  217963 non-null  object 
 13  wm_srsrc     0 non-null       float64
 14  wm_vltyp     217963 non-null  int64  
 15  product_id   217963 non-null  object 
 16  wm_qdocno    1 non-null       float64
 17  wm_tapos     217963 non-null  int64  
dtypes: float64(2), int64(3),

## 1.2 Data Type Conversion

The raw dataset contains several columns stored in incorrect formats (e.g., numeric and date fields saved as text). 

In this step, numeric columns such as `volume` and `wm_vsola` are converted to proper numeric types, and date/time fields (`calday`, `start_time`, `finish_time`) are converted to datetime format. 

This ensures accurate calculations, especially for duration analysis and future aggregations.

In [5]:
df = logistics_data.copy()

# 1️. Fix Volume (comma decimal → dot)
df["volume"] = df["volume"].str.replace(",", ".", regex=False)
df["volume"] = pd.to_numeric(df["volume"], errors="coerce")

# 2️. Convert wm_vsola to numeric
df["wm_vsola"] = pd.to_numeric(df["wm_vsola"], errors="coerce")

# 3️. Convert date columns
df["calday"] = pd.to_datetime(df["calday"], errors="coerce")
df["calmonth"] = pd.to_datetime(df["calmonth"], errors="coerce")
df["calyear"] = pd.to_datetime(df["calyear"], errors="coerce")

# 4️. Convert timestamps
df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
df["finish_time"] = pd.to_datetime(df["finish_time"], errors="coerce")

print("Data types after conversion:\n")
print(df.dtypes)

print("\nMissing values check (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10))

Data types after conversion:

location               object
volume                float64
picker                 object
wm_vsola              float64
area                   object
warehouse              object
calday         datetime64[ns]
calweek                 int64
calmonth       datetime64[ns]
calyear        datetime64[ns]
collli_id              object
start_time     datetime64[ns]
finish_time    datetime64[ns]
wm_srsrc              float64
wm_vltyp                int64
product_id             object
wm_qdocno             float64
wm_tapos                int64
dtype: object

Missing values check (top 10):
wm_srsrc      217963
wm_vsola      217963
wm_qdocno     217962
start_time        94
volume             0
location           0
calday             0
picker             0
area               0
warehouse          0
dtype: int64


## 1.3 Create Picking Duration and Remove Invalid Records


In [6]:
# 1. create duration in seconds
df["duration_sec"] = (df["finish_time"] - df["start_time"]).dt.total_seconds()

print("shape before cleaning", df.shape)

shape before cleaning (217963, 19)


In [7]:
# 2. remove rows with missing start_time
df = df[df["start_time"].notna()]

# 3. remove zero or negative durations
df = df[df["duration_sec"] > 0]

print("shape after cleaning", df.shape)

# Check how many rows were removed
print("Rows removed:", logistics_data.shape[0] - df.shape[0])

shape after cleaning (208089, 19)
Rows removed: 9874


In this step, we calculate the picking duration in seconds using the difference between `finish_time` and `start_time`.

Records with missing timestamps or non-positive durations are removed, as they likely represent system errors or invalid scanning activity. 

This ensures that the dataset reflects only valid picking tasks for accurate analysis.

## 1.4 Fix Invalid Volume Values


In [8]:
#count zero or negative volume values
invalid_volumes = (df["volume"] <= 0).sum()

print("Number of invalid volumes :", invalid_volumes)

Number of invalid volumes : 201


In [9]:
# 1. Set invalid volumes to NaN
df.loc[df["volume"] <= 0, "volume"] = np.nan

# Check missing volume before imputation
print("Missing volume before:", df["volume"].isna().sum())


Missing volume before: 201


In [10]:
# 2. Impute using median volume per product
df["volume"] = df.groupby("product_id") ["volume"]\
                    .transform(lambda x:x.fillna(x.median()))

# Check missing volume after
print("Missing volume after:", df["volume"].isna().sum())

Missing volume after: 0


Volume is a key operational variable influencing picking duration.

Invalid values (zero, negative, or missing) are replaced using the median volume within each product group. This approach preserves product-specific size characteristics, since different products (e.g., apples vs. bananas) naturally vary in volume. Using product-level medians ensures more realistic and operationally meaningful estimates compared to applying a global average.


## 1.5 Create Analytical Features

In [11]:
# 1️. Extract Date from start_time
df["date"] = df["start_time"].dt.date

# 2️. Weekday Name
df["weekday_name"] = df["start_time"].dt.day_name()

# 3️. Weekend Flag
df["is_weekend"] = df["weekday_name"].isin(["Saturday", "Sunday"])

# 4️. Extract Location Components
df["location_row"] = df["location"].str[3:5].astype(int)
df["location_id"] = df["location"].str[6:9].astype(int)

print("New columns added successfully.")

New columns added successfully.


In [12]:
print(df[["date", "weekday_name", "is_weekend", "location_row", "location_id"]].head())

         date weekday_name  is_weekend  location_row  location_id
0  2023-04-01     Saturday        True             6           36
1  2023-04-01     Saturday        True             6           36
2  2023-04-01     Saturday        True             6           36
3  2023-04-01     Saturday        True             6           36
4  2023-04-01     Saturday        True             6           36


Additional analytical fields are created to support business reporting and warehouse modeling.

Date-related fields enable time-based analysis (weekday trends and weekend comparison), while location components allow performance evaluation by warehouse zone. These features will later form the basis of dimension tables in the data warehouse.

In [13]:
print(df.head())

      location  volume    picker  wm_vsola    area    warehouse     calday  \
0  10-06-036-1   0.003  picker_0       NaN  area_0  warehouse_0 2023-04-01   
1  10-06-036-1   0.003  picker_1       NaN  area_0  warehouse_0 2023-04-01   
2  10-06-036-1   0.003  picker_2       NaN  area_0  warehouse_0 2023-04-01   
3  10-06-036-1   0.003  picker_3       NaN  area_0  warehouse_0 2023-04-01   
4  10-06-036-1   0.003  picker_4       NaN  area_0  warehouse_0 2023-04-01   

   calweek   calmonth    calyear  ... wm_vltyp product_id wm_qdocno  wm_tapos  \
0   202313 2023-04-01 2023-01-01  ...       50  product_0       NaN         1   
1   202313 2023-04-01 2023-01-01  ...       50  product_0       NaN         1   
2   202313 2023-04-01 2023-01-01  ...       50  product_0       NaN         1   
3   202313 2023-04-01 2023-01-01  ...       50  product_0       NaN         1   
4   202313 2023-04-01 2023-01-01  ...       50  product_0       NaN         1   

   duration_sec        date  weekday_name  i

## 1.6 Remove Unnecessary Columns
To simplify the dataset for analytical and BI purposes, we remove system-generated and non-informative columns. 

Columns such as `area`, `warehouse`, `wm_srsrc`, `wm_qdocno`, and `wm_tapos` do not contribute to operational performance analysis and are therefore excluded. 

Key operational variables like `product_id`, `collli_id`, `wm_vsola`, and location identifiers are retained to allow deeper analysis if needed.

In [14]:
# Drop columns that are not useful for BI/analytics
columns_to_drop = [
    "area",
    "warehouse",
    "calweek",
    "calmonth",
    "calyear",
    "wm_tapos",
    "wm_srsrc",   # all missing
    "wm_qdocno"   # almost all missing
]

df = df.drop(columns=columns_to_drop)

print("Remaining columns:")
print(df.columns.tolist())


Remaining columns:
['location', 'volume', 'picker', 'wm_vsola', 'calday', 'collli_id', 'start_time', 'finish_time', 'wm_vltyp', 'product_id', 'duration_sec', 'date', 'weekday_name', 'is_weekend', 'location_row', 'location_id']


## 1.7 Export Clean Analytical Dataset

After cleaning and removing unnecessary columns, the dataset is ready for analytical use.

The cleaned dataset will serve as the staging layer for SQL data warehouse modeling and Power BI reporting. This ensures consistent and reproducible data transformations.

In [15]:
# Final dataset shape
print("Final dataset shape:", df.shape)

# Save clean dataset inside data folder
df.to_csv("data/clean_picking_data.csv", index=False)

print("Clean dataset saved in data/ folder")

Final dataset shape: (208089, 16)
Clean dataset saved in data/ folder


## 1.8 Connect to database

In [23]:
#install packages
!pip install sqlalchemy psycopg2-binary pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
# connect jupyter to PostgreSQL
from sqlalchemy import create_engine

# Databse connection details
username = os.getenv("PG_USER")
password = os.getenv("PG_PASSWORD")
host = os.getenv("PG_HOST")
port = os.getenv("PG_PORT")
database = os.getenv("PG_DATABASE")

# Create PostgreSQL engine
engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# test connection
with engine.connect() as conn:
    print(" Connected to PostgreSQL!")

 Connected to PostgreSQL!


In [25]:
# load dataframe to Postgres
df = pd.read_csv("data/clean_picking_data.csv")

# Load into public schema/table (auto-creates table)
df.to_sql(
    "clean_picking_data",
    engine,
    schema="staging",
    if_exists="append", # as table already exist
    index=False
)

print("Loaded into PostgreSQL: staging.clean_picking_data")

Loaded into PostgreSQL: staging.clean_picking_data


The cleaned dataset is loaded into PostgreSQL to establish a structured data warehouse environment. This enables SQL-based modeling and prepares the data for dimensional design and BI reporting.